## build_fact_realtor
Rebuilds `gold.fact_realtor_metro_monthly` from `silver.fact_realtor_metro_monthly`: carries the 15 metrics verbatim (the Sep-Nov 2022 methodology-break note already lives in the G0 column COMMENTs). No derived columns. Full rebuild via `INSERT OVERWRITE`. Spec: `gold_layer_design.md` §3.5.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
STEP_SEQUENCE = 5                       # position owned by the orchestrator (G4)
SOURCE_TABLE  = f"{SILVER}.fact_realtor_metro_monthly"
TARGET_TABLE  = f"{GOLD}.fact_realtor_metro_monthly"
DIM_GEO       = f"{GOLD}.dim_geo"
DIM_DATE      = f"{GOLD}.dim_date"

# The 15 source measures, IN G0 COLUMN ORDER (gold_ddl.py). Carried at native precision.
MEASURES = [
    "median_listing_price", "active_listing_count", "median_days_on_market", "new_listing_count",
    "price_increased_count", "price_increased_share", "price_reduced_count", "price_reduced_share",
    "pending_listing_count", "median_listing_price_per_square_foot", "median_square_feet",
    "average_listing_price", "total_listing_count", "pending_ratio", "quality_flag",
]

In [ ]:
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "gold",
    target_table    = TARGET_TABLE,
)
print(f"build_fact_realtor: step_log_id={step.step_log_id}")

In [ ]:
# Carry the 15 metrics verbatim; set fresh Gold audit timestamps.
try:
    src = spark.table(SOURCE_TABLE)
    rows_read = src.count()
    staged = src.select(
        "geo_key", "date_key",
        *[F.col(col_name) for col_name in MEASURES],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    )
    staged.createOrReplaceTempView("gold_fact_realtor_staging")
    step.rows_read = rows_read
    print(f"build_fact_realtor: read {rows_read:,} Silver rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Full rebuild via INSERT OVERWRITE (design §2.2) — preserves G0 schema/PK/FK/COMMENTs.
# FK is informational (not enforced), so check geo_key/date_key resolve in the Gold dims here.
transform_started = datetime.now(timezone.utc)
try:
    staged = spark.table("gold_fact_realtor_staging")
    geo_orphans  = staged.join(spark.table(DIM_GEO).select("geo_key"),  "geo_key",  "left_anti").count()
    date_orphans = staged.join(spark.table(DIM_DATE).select("date_key"), "date_key", "left_anti").count()
    if geo_orphans or date_orphans:
        raise AssertionError(f"[{TARGET_TABLE}] FK orphans: geo={geo_orphans:,} date={date_orphans:,}")

    spark.sql(f"INSERT OVERWRITE TABLE {TARGET_TABLE} SELECT * FROM gold_fact_realtor_staging")

    post_count = spark.table(TARGET_TABLE).count()
    if post_count != step.rows_read:
        raise AssertionError(f"[{TARGET_TABLE}] Row-count mismatch: read {step.rows_read:,}, wrote {post_count:,}.")
    step.rows_written = post_count
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=post_count, rows_inserted=post_count, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"build_fact_realtor: wrote {post_count:,} rows to {TARGET_TABLE}")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        error_message=f"{type(e).__name__}: {e}", ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise